# Martingale Verification Generator

Notebook workbench for generating prompts, reasoning traces, and canonical answers for martingale-verification problems.

In [1]:
from pathlib import Path
import json
import sys

project_root = Path.cwd()
while project_root != project_root.parent and not (project_root / "benchmark").exists():
    project_root = project_root.parent

if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

In [2]:
from benchmark.generators import MartingaleVerificationGenerator

gen = MartingaleVerificationGenerator()

In [3]:
params = gen.sample_params(seed=101, difficulty=2, split="dev")
params

{'problem_type': 'quadratic_compensation',
 'difficulty': 2,
 'split': 'dev',
 'seed': 101,
 'process_notation': 'S',
 'time_notation': 'n',
 'step': 5,
 'variance': 25,
 'compensation': 24,
 'surface_variant': 'variance_given',
 'is_martingale': False,
 'reason_code': 'wrong_compensation'}

In [4]:
problem = gen.generate_problem(params)
reasoning = gen.generate_reasoning(params)
solution = gen.generate_solution(params)

print("PROBLEM:\n", problem)
print("\nREASONING:\n", reasoning)
print("\nSOLUTION:\n", gen.to_json_safe(solution))

PROBLEM:
 Let S_n = Y_1 + ... + Y_n for independent increments with the increments have mean 0 and variance 25. Consider M_n = S_n^2 - 24 n. With respect to the natural filtration, is (M_n) a martingale? Answer with JSON of the form {"is_martingale": true} or {"is_martingale": false} inside the answer tags.

REASONING:
 Use the quadratic martingale for centered independent increments: if S_n = S_0 + Y_1 + ... + Y_n, the increments have mean 0 and variance sigma^2, and the increments are independent of the past, then M_n = S_n^2 - n sigma^2 is a martingale. For this process, E[S_(n+1)^2 | F_n] = S_n^2 + Var(Y_(n+1)). Here Var(Y_(n+1)) = 25. Therefore S_n^2 - c n is a martingale exactly when c = 25. The proposed compensation is c = 24, which does not equal the variance.

Final answer:
<answer>
{"is_martingale": false}
</answer>

SOLUTION:
 {'is_martingale': False}


In [5]:
records = []
for difficulty in [1, 2, 3]:
    for seed in range(1000 + 100 * difficulty, 1005 + 100 * difficulty):
        records.append(gen.generate_record(seed=seed, difficulty=difficulty, split="dev"))

len(records), records[0]

(15,
 {'id': 'martingale_verification_dev_001100',
  'family': 'martingale_verification',
  'problem_type': 'centered_walk_basic',
  'difficulty': 1,
  'split': 'dev',
  'seed': 1100,
  'params': {'problem_type': 'centered_walk_basic',
   'difficulty': 1,
   'split': 'dev',
   'seed': 1100,
   'process_notation': 'S',
   'time_notation': 'n',
   'step': 2,
   'is_martingale': True,
   'reason_code': 'zero_drift'},
  'problem': 'Let S_0 = 0 and S_n = Y_1 + ... + Y_n, where the Y_k are independent and P(Y_k = 2) = P(Y_k = -2) = 1/2. With respect to the natural filtration, is (S_n) a martingale? Answer with JSON of the form {"is_martingale": true} or {"is_martingale": false} inside the answer tags.',
  'reasoning': 'Use the conditional expectation martingale test: an adapted integrable process (M_n) is a martingale with respect to (F_n) if E[M_(n+1) | F_n] = M_n for every n. Here the process is adapted to the natural filtration. Since E[Y_(n+1)] = 0 and Y_(n+1) is independent of the natur

In [6]:
output_path = project_root / "benchmark" / "data" / "dev" / "martingale_verification_preview.jsonl"
with output_path.open("w") as f:
    for record in records:
        f.write(json.dumps(record, sort_keys=True) + "\n")

output_path

PosixPath('/Users/xingzheli/Documents/Python-WorkSpace/SC_fine-tune/benchmark/data/dev/martingale_verification_preview.jsonl')